In [1]:
# =========================================
# CHILD RUNNING TEST (DATA-DRIVEN, NO HARD ASSUMPTIONS)
# =========================================

import cv2
import numpy as np
import mediapipe as mp
from scipy.signal import find_peaks
from ultralytics import YOLO

In [2]:
# -------------------------------
# MediaPipe
# -------------------------------
mpPose = mp.solutions.pose
pose = mpPose.Pose(min_detection_confidence=0.6)
mpDraw = mp.solutions.drawing_utils

DRAW_LM = mpDraw.DrawingSpec(color=(0,0,255), thickness=2, circle_radius=2)
DRAW_CONN = mpDraw.DrawingSpec(color=(0,0,0), thickness=2)

In [3]:
# -------------------------------
# Utils
# -------------------------------
def safe_mean(x): return float(np.mean(x)) if len(x) else 0.0
def safe_std(x): return float(np.std(x)) if len(x) else 0.0

def smooth(x, k=5):
    if len(x) < k: return np.array(x)
    return np.convolve(x, np.ones(k)/k, mode='same')

def angle(a,b,c):
    a,b,c = np.array(a),np.array(b),np.array(c)
    ba, bc = a-b, c-b
    cos = np.dot(ba,bc)/(np.linalg.norm(ba)*np.linalg.norm(bc)+1e-6)
    return np.degrees(np.arccos(np.clip(cos,-1,1)))

In [4]:
# -------------------------------
# Cone detection (optional)
# -------------------------------
def detect_cones(path):
    try:
        model = YOLO("cones_best.pt")
    except:
        return None
    cap = cv2.VideoCapture(path)
    centers=[]
    while True:
        ret,f=cap.read()
        if not ret: break
        r=model(f,conf=0.5,verbose=False)
        for b in r[0].boxes:
            x1,y1,x2,y2=map(int,b.xyxy[0])
            centers.append(((x1+x2)//2,(y1+y2)//2))
    cap.release()
    return centers if len(centers)>=2 else None

In [5]:
# Arm-Leg Opposition
def arm_score_cal(corr):
    arm_score = 0
    if corr < -0.7: arm_score=5
    elif corr < -0.5: arm_score=4
    elif corr < -0.3: arm_score=3
    elif corr < -0.1: arm_score=2
    else: arm_score=1
    return arm_score

In [6]:
# Flight Phase
def flight_score_cal(flight_ratio):
    flight_score = 0
    if flight_ratio > 0.6: flight_score=5
    elif flight_ratio > 0.45: flight_score=4
    elif flight_ratio > 0.3: flight_score=3
    elif flight_ratio > 0.1: flight_score=2
    else: flight_score=1
    return flight_score

In [7]:
# Foot Placement (relative)
def foot_score_cal(foot_score_val):
    foot_score = 0
    if foot_score_val < 90: foot_score=5
    elif foot_score_val < 110: foot_score=4
    elif foot_score_val < 130: foot_score=3
    elif foot_score_val < 150: foot_score=2
    else: foot_score=1
    return foot_score

In [8]:
# Leg Flexion
def leg_score_cal(knee_val):
    leg_score = 0
    if knee_val < 90: leg_score=5
    elif knee_val < 110: leg_score=4
    elif knee_val < 130: leg_score=3
    elif knee_val < 150: leg_score=2
    else: leg_score=1
    return leg_score

In [9]:
# Trajectory
def traj_score_cal(traj_error):
    traj_score = 0
    if traj_error < 0.005: traj_score=5
    elif traj_error < 0.01: traj_score=4
    elif traj_error < 0.02: traj_score=3
    elif traj_error < 0.04: traj_score=2
    else: traj_score=1
    return traj_score

In [10]:
def threshold_line(path):
    
    model = YOLO("best.pt")
    cap = cv2.VideoCapture(path)

    left_x, right_x = [], []
    frame_count = 0
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        results = model(frame, conf=0.5, verbose=False)

        centers = []
        for box in results[0].boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            conf = box.conf.item()
            cx = (x1 + x2) // 2
            centers.append(cx)
            
            # Draw rectangle on cone
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

            # Optional: confidence label
            cv2.putText(frame, f"Cone {conf:.2f}", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

        if len(centers) >= 2:
            centers.sort()
            left_x.append(centers[0])
            right_x.append(centers[-1])

        if len(left_x) > 50:
            break

        cv2.namedWindow("Video", cv2.WINDOW_NORMAL)
        cv2.setWindowProperty("Video", cv2.WND_PROP_FULLSCREEN, cv2.WINDOW_FULLSCREEN)
        cv2.imshow("Video", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

    # Final fixed threshold
    left_avg = int(np.mean(left_x))
    right_avg = int(np.mean(right_x))
    # threshold_x = int((left_avg + right_avg) / 2)

    print("Left cone X:", left_avg)
    print("Right cone X:", right_avg)
    # print("Threshold X:", threshold_x)
    return left_avg, right_avg


In [14]:
# MAIN
# -------------------------------
def running_test(path="video.mp4"):

    cap=cv2.VideoCapture(path)
    fps=cap.get(cv2.CAP_PROP_FPS)

    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    #threshold line calculate  
    left_line_x, right_line_x = 0.10, 0.85
    left_line_x, right_line_x = threshold_line(path)
    left_line_x = round((left_line_x/frame_width), 2) 
    right_line_x = round((right_line_x/frame_width), 2) 
    print(f"{left_line_x} --- {right_line_x}")

    cones = detect_cones(path)  # used only for visualization if available

    hip_x, hip_y = [], []
    l_ank_y, r_ank_y = [], []
    l_el_y, r_el_y = [], []
    l_knee_ang, r_knee_ang = [], []
    vis_frames = []

    frame_idx=0
    window="Running Analysis"
    cv2.namedWindow(window, cv2.WINDOW_NORMAL)

    while True:
        ret,frame=cap.read()
        if not ret: break

        rgb=cv2.cvtColor(frame,cv2.COLOR_BGR2RGB)
        res=pose.process(rgb)

        if res.pose_landmarks:
            lms=res.pose_landmarks.landmark

            # hip center
            hx = (lms[23].x + lms[24].x) / 2

            # boundary check
            if not (left_line_x <= hx <= right_line_x):
                continue

            # visibility filter
            if lms[23].visibility < 0.5: 
                continue

            mpDraw.draw_landmarks(frame,res.pose_landmarks, mpPose.POSE_CONNECTIONS,DRAW_LM,DRAW_CONN)

            # key points
            l_sh=(lms[11].x,lms[11].y); r_sh=(lms[12].x,lms[12].y)
            l_el=(lms[13].x,lms[13].y); r_el=(lms[14].x,lms[14].y)
            l_hip=(lms[23].x,lms[23].y); r_hip=(lms[24].x,lms[24].y)
            l_knee=(lms[25].x,lms[25].y); r_knee=(lms[26].x,lms[26].y)
            l_ank=(lms[27].x,lms[27].y); r_ank=(lms[28].x,lms[28].y)

            # COM
            hx=(l_hip[0]+r_hip[0])/2
            hy=(l_hip[1]+r_hip[1])/2
            hip_x.append(hx); hip_y.append(hy)

            # ankles + arms
            l_ank_y.append(l_ank[1]); r_ank_y.append(r_ank[1])
            l_el_y.append(l_el[1]); r_el_y.append(r_el[1])

            # knee flexion
            l_knee_ang.append(angle(l_hip,l_knee,l_ank))
            r_knee_ang.append(angle(r_hip,r_knee,r_ank))

            vis_frames.append(frame_idx)

        # show
        cv2.imshow(window,frame)
        if cv2.waitKey(1)&0xFF==27: break

        frame_idx+=1

    cap.release(); cv2.destroyAllWindows()

    # -------------------------------
    # PREPROCESS
    # -------------------------------
    hip_x = smooth(hip_x)
    hip_y = smooth(hip_y)
    l_ank_y = smooth(l_ank_y)
    r_ank_y = smooth(r_ank_y)

    n = min(len(hip_x), len(l_ank_y), len(r_ank_y))
    hip_x, hip_y = hip_x[:n], hip_y[:n]
    l_ank_y, r_ank_y = l_ank_y[:n], r_ank_y[:n]
    l_el_y, r_el_y = l_el_y[:n], r_el_y[:n]

    if n < 10:
        return 1,1,1,1,1  # insufficient data

    # -------------------------------
    # STEP SEGMENTATION
    # -------------------------------
    peaks,_ = find_peaks(-l_ank_y, distance=5)
    peaks = peaks[peaks < n]

    # -------------------------------
    # FLIGHT PHASE (velocity-based)
    # -------------------------------
    vel_l = np.diff(l_ank_y)
    vel_r = np.diff(r_ank_y)

    contact_l = (l_ank_y[1:] > np.percentile(l_ank_y,70)) & (np.abs(vel_l)<0.01)
    contact_r = (r_ank_y[1:] > np.percentile(r_ank_y,70)) & (np.abs(vel_r)<0.01)

    flight = ~(contact_l | contact_r)
    flight_ratio = np.mean(flight)

    # -------------------------------
    # ARM-LEG OPPOSITION (correlation)
    # -------------------------------
    min_len=min(len(l_el_y),len(r_ank_y))
    corr = np.corrcoef(l_el_y[:min_len], r_ank_y[:min_len])[0,1]

    # -------------------------------
    # FOOT PLACEMENT (relative angle)
    # -------------------------------
    foot_angles = np.array(l_knee_ang[:n])
    foot_score_val = safe_mean(foot_angles)

    # -------------------------------
    # LEG FLEXION
    # -------------------------------
    knee_val = safe_mean(l_knee_ang + r_knee_ang)

    # -------------------------------
    # TRAJECTORY (line fit error)
    # -------------------------------
    t_idx = np.arange(len(hip_x))
    coef = np.polyfit(t_idx, hip_x, 1)
    fit = np.polyval(coef, t_idx)
    traj_error = safe_mean(np.abs(hip_x - fit))

    # =========================================================
    # SCORING (LIKERT)
    # =========================================================

    # Arm-Leg Opposition
    arm_score = arm_score_cal(corr)

    # Flight Phase
    flight_score = flight_score_cal(flight_ratio)

    # Foot Placement (relative)
    foot_score = foot_score_cal(foot_score_val) 

    # Leg Flexion
    leg_score = leg_score_cal(knee_val)

    # Trajectory
    traj_score = traj_score_cal(traj_error)

    print("------ RESULT ------")
    print("Arm-Leg:", arm_score)
    print("Flight:", flight_score)
    print("Foot:", foot_score)
    print("Leg:", leg_score)
    print("Trajectory:", traj_score)

    final_score = (arm_score + flight_score + foot_score + leg_score + traj_score)/5

    return final_score

In [15]:
path = "data/running.mp4"
print(running_test(path))

Left cone X: 64
Right cone X: 688
0.08 --- 0.81
------ RESULT ------
Arm-Leg: 2
Flight: 4
Foot: 1
Leg: 1
Trajectory: 1
1.8
